# Health checks

Blueprints have **checks** that look for problems in the data: gaps, flat values, implausible values and outliers. `device.health_checks()` runs them and gives:

- `device.health`: a summary as plain json (what Smart Citizen Flows stores after each processing run)
- `device.qc_data`: the data with a flag column per check and column, and optionally the data without the flagged values

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

In [ ]:
import pandas as pd

device = sc.Device(params=sc.APIParams(id=18154))
device.options.min_date = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('3D')).strftime('%Y-%m-%d')
await device.load()
device.process()

## The checks of the blueprint

Each check has a function in `scdata.device.check` and its settings. For instance, a gap is a period without readings of a column longer than its gap size (5 minutes, 10 for the SEN5X, which reads every 5 minutes).

In [ ]:
for check in device.checks:
    print(check.name, '-', check.function, '- clean:', check.clean, '- settings:', list(check.kwargs or {}))

In [ ]:
next(check for check in device.checks if check.name == 'GAPS').kwargs

## Run the checks

The checks of `sc_air` cover all the sensors a kit can have: a device without some of them gets a warning per missing column. Here only errors are shown.

In [ ]:
import warnings

set_logger_level('ERROR')
warnings.simplefilter('ignore', pd.errors.PerformanceWarning)

device.health_checks()

### Summary

Per check: its status, and per column the readings flagged (or, for gaps, the time in gaps), the ratio, and the first 10 periods flagged.

In [ ]:
{key: value for key, value in device.health.items() if key != 'checks'}

In [ ]:
[(check['name'], check['status'], len(check['columns'])) for check in device.health['checks']]

In [ ]:
# Ratio flagged per column and check
ratios = pd.DataFrame({check['name']: {column: summary['ratio'] for column, summary in check['columns'].items()}
                       for check in device.health['checks']})
ratios[(ratios > 0).any(axis=1)].sort_values('GAPS', ascending=False)

In [ ]:
# The column with most time in gaps
gaps = next(check for check in device.health['checks'] if check['name'] == 'GAPS')
column, summary = max(gaps['columns'].items(), key=lambda item: item[1]['ratio'])
column, summary

The summary is json: it can be stored or sent as it is.

In [ ]:
import json

len(json.dumps(device.health))

### Flags

`device.qc_data` has the data and a boolean column per check and column (`<CHECK>__<column>`):

In [ ]:
flags = [column for column in device.qc_data.columns if column.startswith(('GAPS__', 'IMPLAUSIBLE__', 'FLAT__', 'OUTLIERS__'))]
device.qc_data[flags].sum().sort_values(ascending=False).head(10)

`device.quality_metrics` has the share of rows flagged per check and column (for the checks with `store_qc`):

In [ ]:
dict(list(device.quality_metrics.items())[:5])

## Clean data and statistics

Checks with `clean` replace the flagged values with NaN in `qc_data`. With `resample`, the checks run on averages (for example, of 10 minutes), and `get_min` and `get_max` add the minimum and maximum of each period (`MIN__<column>`, `MAX__<column>`).

In [ ]:
for check in device.checks:
    check.clean = check.name in ('IMPLAUSIBLE', 'OUTLIERS')

device.health_checks(resample='10Min', get_min=True, get_max=True)
device.qc_data[['NOISE_A', 'MIN__NOISE_A', 'MAX__NOISE_A', 'CLEAN__NOISE_A']].head()

Export the quality data, one file per export of the blueprint:

In [ ]:
from os.path import join
from scdata._config import config

device.export(join(config.paths['export'], 'examples'), qc_data=True, forced_overwrite=True)

## Change the settings

Checks are models: change their settings, or add checks, before running them.

In [ ]:
from scdata.models import Check

# Gaps of 30 minutes or more only
next(check for check in device.checks if check.name == 'GAPS').kwargs['default_gap_size_minutes'] = 30

# Noise above 85 dBA, as implausible values
device.checks.append(Check(name='LOUD', description='Noise above 85 dBA', function='find_implausible_values',
                           kwargs={'implausible_values': [{'column': 'NOISE_A', 'limits': [0, 85]}]}))
device.health_checks()
[(check['name'], check['status'], check['columns'].get('NOISE_A', {}).get('flagged')) for check in device.health['checks']]

## Several devices

The same summary for a group of devices, to compare them:

In [ ]:
summaries = {}
for device_id in [18154, 17592]:
    item = sc.Device(params=sc.APIParams(id=device_id))
    item.options.min_date = device.options.min_date
    await item.load()
    item.process()
    item.health_checks()
    summaries[device_id] = {check['name']: max([summary['ratio'] for summary in check['columns'].values()], default=0)
                            for check in item.health['checks']}

# Highest ratio flagged by each check
pd.DataFrame(summaries).T